# पाठ ०५ - एजेन्टिक RAG


## सेटअप

यो नोटबुकले Microsoft Agent Framework प्रयोग गरेर Agentic RAG (Retrieval-Augmented Generation) ढाँचा देखाउँछ।

**पूर्वआवश्यकताहरू:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — तपाईंको Azure AI Search सेवा अन्तिम बिन्दु
- `AZURE_SEARCH_API_KEY` — तपाईंको Azure AI Search API कुञ्जी
- वातावरण चरहरूको माध्यमबाट कन्फिगर गरिएको Azure OpenAI परिनियोजन
- Azure CLI प्रमाणित गरिएको (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## एजेन्टिक RAG के हो?

परम्परागत RAG एउटा निश्चित पाइपलाइन पालना गर्छ: कागजातहरू प्राप्त गर्नुहोस्, त्यसपछि प्रतिक्रिया तयार गर्नुहोस्। **एजेन्टिक RAG** ले अझ अगाडि बढेर एजेन्टलाई **कहिले** र **कसरी** जानकारी प्राप्त गर्ने निर्णय गर्ने स्वतन्त्रता दिन्छ।

एजेन्टिक RAG सँग, एजेन्टले सक्दछ:
- प्रश्नको उत्तर दिनु अघि प्राप्ति आवश्यक छ कि छैन भनेर **निर्णय लिन**
- कुन डाटा स्रोत वा उपकरण प्रश्न गर्न **छान्नु**
- प्राप्त नतिजाहरूलाई **मूल्याकन गर्नु** र यदि पहिलो प्रयास अपर्याप्त भएमा थप प्राप्ति प्रक्रिया गर्नु
- धेरै प्राप्ति चरणहरूबाट जानकारीलाई एक सुसंगत उत्तरमा **संयोजन गर्नु**

यसले एजेन्टलाई स्थिर प्राप्ति-पछि-उत्पादन पाइपलाइनको तुलनामा बढी लचकदार र सही बनाउँछ।


## खोज उपकरण सिर्जना गर्नुहोस्

Agentic RAG मा, बाह्य डाटा स्रोतहरू **उपकरण** रूपमा आवृत्त गरिन्छ जसलाई एजेन्ट आवश्यक परेमा बोलाउन सक्छ। यसले एजेन्टलाई प्राप्तिलाई एउटा अनिवार्य चरणको सट्टा अर्को क्रिया मात्रको रूपमा लिन अनुमति दिन्छ।

तल हामी यात्रा ज्ञान आधार परिभाषित गर्छौं र यसलाई एउटा उपकरणको रूपमा सार्वजनिक गर्छौं कि एजेन्ट गन्तव्य जानकारी खोज्न यसलाई बोलाउन सक्छ।


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG एजेन्ट निर्माण गर्दै

अब हामी एउटा एजेन्ट बनाउछौं जुनलाई **सधैं उत्तर दिनुअघि सूचना प्राप्त गर्न निर्देशन दिइएको हुन्छ**। एजेन्टले आफ्नो प्रतिक्रिया ज्ञान आधारमा आधारित बनाउन `search_travel_knowledge` उपकरण प्रयोग गर्दछ, आफ्नै तालिम डाटामा भर नपर्ने।


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## पुनरावृत्ति पुनःप्राप्ति — मेकर-चेकर ढाँचा

Agentic RAG को एक प्रमुख फाइदा भनेको **पुनरावृत्ति पुनःप्राप्ति** हो। एजेन्टले यसको प्रारम्भिक निष्कर्षहरूलाई प्रमाणीकरण, सुधार वा विस्तार गर्न धेरै चरणमा खोजी गर्न सक्छ — "मेककर-चेकर" कार्यप्रवाह जस्तै:

1. **मेककर चरण**: एजेन्टले प्रारम्भिक जानकारी ल्याउँछ र जवाफको मसौदा तयार गर्दछ।
2. **चेकर चरण**: एजेन्टले विवरणहरू प्रमाणित गर्न वा खाली ठाउँ भर्न थप पुनःप्राप्ति गर्छ।

तल, एजेन्टलाई यस्तो प्रश्न सोधिएको छ जसले धेरै गन्तव्यहरूको तुलना गर्न आवश्यक हुन्छ, जसले यसलाई धेरै पटक खोजी गर्न प्रोत्साहित गर्दछ।


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## सारांश

यस पाठमा तपाईंले सिक्नुभयो कि कसरी Microsoft Agent Framework प्रयोग गरेर **Agentic RAG** प्रणाली बनाउने:

- **Agentic RAG** ले एजेन्टहरूलाई स्वतन्त्र रूपमा कहिले जानकारी प्राप्त गर्ने निर्णय गर्न अनुमति दिन्छ, जसले प्राप्तिलाई निश्चित सट्टा गतिशील बनाउँछ।
- **टुलहरूलाई डाटा स्रोतको रूपमा प्रयोग गर्ने**: बाह्य ज्ञान आधारहरू (जस्तै Azure AI Search) टुलको रूपमा ढाकिन्छन् जुन एजेन्टले प्रयोग गर्न सक्छ।
- **पुनरावृत्तिगरि प्राप्ति**: मेकर-चेकर ढाँचाले एजेन्टलाई धेरै पुनरावृत्ति चक्रहरू सञ्चालन गर्न सक्षम बनाउँछ — खोज्ने, प्रमाणित गर्ने, र सुधार गर्ने — अन्तिम उत्तर दिने अघि।

उत्पादनमा, तपाईंले मेमोरीमा रहेको `TRAVEL_KNOWLEDGE_BASE` लाई वास्तविक Azure AI Search इन्डेक्ससँग प्रतिस्थापन गर्नुहुनेछ ठूलो पैमानेमा यात्रा दस्तावेज प्राप्त गर्नका लागि।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
